# Project Workflows

1. **Authentication**: a loop that asks for credentials, validates them, and retries on failure.
2. **Question answering**: validates a question and answers it directly using an LLM (halting early on empty input).

In [ ]:
%pip install -q langgraph==0.2.57 langchain-ibm==0.3.10

## 1. Authentication workflow

Flow: `InputNode` -> `ValidateCredential` -> (`Success` -> END) or (`Failure` -> back to `InputNode`).

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END


class AuthState(TypedDict):
    username: Optional[str]
    password: Optional[str]
    is_authenticated: Optional[bool]
    output: Optional[str]

In [ ]:
def input_node(state):
    # Ask for the username only if it isn't already set
    username = state.get('username', '')
    if username == '':
        username = input('What is your username? ')

    password = input('Enter your password: ')
    return {'username': username, 'password': password}


def validate_credentials_node(state):
    username = state.get('username', '')
    password = state.get('password', '')

    # Simulated credential check
    is_authenticated = username == 'test_user' and password == 'secure_password'
    return {'is_authenticated': is_authenticated}


def success_node(state):
    return {'output': 'Authentication successful! Welcome.'}


def failure_node(state):
    return {'output': 'Not successful, please try again!'}


def router(state):
    # Decide which node runs after validation
    if state['is_authenticated']:
        return 'success_node'
    return 'failure_node'

In [ ]:
auth_workflow = StateGraph(AuthState)

auth_workflow.add_node('InputNode', input_node)
auth_workflow.add_node('ValidateCredential', validate_credentials_node)
auth_workflow.add_node('Success', success_node)
auth_workflow.add_node('Failure', failure_node)

auth_workflow.add_edge('InputNode', 'ValidateCredential')
auth_workflow.add_edge('Success', END)
auth_workflow.add_edge('Failure', 'InputNode')  # retry on failure

auth_workflow.add_conditional_edges(
    'ValidateCredential',
    router,
    {'success_node': 'Success', 'failure_node': 'Failure'},
)

auth_workflow.set_entry_point('InputNode')

auth_app = auth_workflow.compile()

In [ ]:
# The simulated credentials are: test_user / secure_password
result = auth_app.invoke({'username': 'test_user'})
print(result['output'])

## 2. Question-answering workflow

Flow: InputNode (validate the question) -> QANode (LLM answer) -> END (or early termination if invalid).

The LLM is IBM Granite on watsonx.ai. Set `WATSONX_API_KEY` and `WATSONX_PROJECT_ID` as environment variables before running.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain_ibm import ChatWatsonx
import os

llm = ChatWatsonx(
    model_id='ibm/granite-3-3-8b-instruct',
    url='https://us-south.ml.cloud.ibm.com',
    project_id=os.environ['WATSONX_PROJECT_ID'],
    api_key=os.environ['WATSONX_API_KEY'],
)

In [ ]:
# 1. State definition
class GeneralQAState(TypedDict):
    question: Optional[str]
    answer: Optional[str]
    valid: Optional[bool]
    error: Optional[str]

# 2. Node: Input validation
def input_validation_node(state: GeneralQAState):
    question = (state.get("question") or "").strip()
    if not question:
        return {"valid": False, "error": "Question cannot be empty."}
    return {"valid": True}

# 3. Router: Halt if input is invalid
def validation_router(state: GeneralQAState):
    if not state.get("valid"):
        return END
    return "QANode"

# 4. Node: Direct LLM query
def general_llm_node(state: GeneralQAState):
    question = state.get("question", "")
    
    prompt = (
        "You are a helpful, accurate, and direct assistant. "
        f"Answer the following question:\n\n{question}"
    )
    
    try:
        response = llm.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error occurred: {str(e)}"}

In [ ]:
# 5. Build Graph
qa_workflow = StateGraph(GeneralQAState)
qa_workflow.add_node("InputNode", input_validation_node)
qa_workflow.add_node("QANode", general_llm_node)
qa_workflow.set_entry_point("InputNode")

qa_workflow.add_conditional_edges(
    "InputNode",
    validation_router,
    {"QANode": "QANode", END: END}
)

qa_workflow.add_edge("QANode", END)

qa_app = qa_workflow.compile()

In [ ]:
# 1. Valid question
response = qa_app.invoke({"question": "What is the capital of France?"})
print("Answer:", response["answer"])

# 2. Validation check (empty question)
invalid_run = qa_app.invoke({"question": ""})
print("Valid?", invalid_run.get("valid"))
print("Error:", invalid_run.get("error"))
print("Answer:", invalid_run.get("answer"))  # Should be None because it bypassed QANode